In [ ]:
###!/usr/bin/env python
################################################
# New style 
# ###############################################
import sys

rootdir_ = '../'
if ( rootdir_ not in sys.path ):
    sys.path.append(rootdir_)
    print( f" a path to {rootdir_} added in {__name__} ")


from Utils import GridUtils as GrU
from Utils import MakePressures as MkP
from Utils import utils as uti
from Utils import MyConstants as Co
from Utils import time_utils as tuti
from Utils import numerical_utils as nuti

import analysis_utils as auti
import file_utils as futi
import event_utils as euti


#from PyRegridding.Utils import MakePressures as MkP
#from Drivers import RegridField as RgF
import RegridField as RgF

# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# for smoothing , nonlienar colors ...
from scipy.ndimage import uniform_filter
from scipy.ndimage import gaussian_filter
import matplotlib.colors as mcolors

# Some other useful packages 
import copy
import time
import cftime
import yaml
import numbers

# Some other useful packages 
import importlib
from pathlib import Path


importlib.reload( auti )
importlib.reload( futi )
importlib.reload( euti )

Rdair=Co.Rdair()


In [ ]:
# This allow both dict.key and dict['key'] syntax
class AttrDict(dict):
    def __getattr__(self, key):
        try:
            return self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")

    def __setattr__(self, key, value):
        self[key] = value

    def __delattr__(self, key):
        try:
            del self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")



In [ ]:
%%time
nsteps=None
start_date=None
#super_lat_range = [-90.,90.]  #[-85,-30]
super_lat_range = [-80.,-30.]  #[-85,-30]
#case, process_ncdata, start_date, nsteps = 'c153_topfix_ne240pg3_FMTHIST_xic_x02'   , False, [2004,7,15,0], 248
#case, process_ncdata  = 'cam77_dyamond1_prod1'    , False
case, process_ncdata  = 'c124_dyamond1_prod2'    , False
#case , process_ncdata = 'xy-rdg-mm-front'    , True
A = futi.read_case( case=case, nsteps=nsteps, start_date=start_date , super_lat_range=super_lat_range ) # , nsteps = 31*8 )

time, zlev, lat, lon = A.time, A.zlev, A.lat, A.lon



In [ ]:
#################################################################
# Make event lists ... and composites
importlib.reload(euti)
importlib.reload(auti)

#zlev_event=10_000. #23_000.
zlev_event=15_000. #23_000.
lat_range=  [-50,-40] #[-70,-60] #[-60,-40]
lat_range=  [-60,-40] #[-70,-60] #[-60,-40]
lon_range=[0,360] # [0,60]
exclude_orography=True
topo_thresh=1.e-12 # needs to be this low to remove Malvinas/Falklands(?)  ... 0.0001 leaves them. 
###############################################
# Scope out range and distribution of events
###############################################
thresh=[0.,1e6]
second_thresh=None #second_thresholds[ithr]
ds =euti.make_ds(fld=A.rho_epwp[:,:,:,:], lon=lon, lat=lat, zlev=zlev, time=time, 
                 thresh=thresh,second_thresh=second_thresh,zlev_event=zlev_event, 
                 lat_range=lat_range, lon_range=lon_range)

# get shape of varaiables
nt,nz,ny,nx = np.shape( A.u )

htopo_t = np.tile(A.htopo[None, :, :], ( nt, 1, 1))
htopo_4D_x , time4D,lat4D,lon4D  = auti.cube4D_ds( event_ds=ds, aa=htopo_t , lon=lon, lat=lat, window=[0,5,5] , TZHkey='tyx', lat_range=[-999,999], lon_range=[-999,999] )

htopo_MMM = auti.collapseSpace( htopo_4D_x , TZHkey='etyx')
htopo_super_max = htopo_MMM[2].max( axis=1 )


########################################
# Exclude events with topography nearby
########################################
if (exclude_orography==True):
    flat=np.where(htopo_super_max<1.e-12 )
    flat[0].shape
    ds_flat=ds.isel( index=flat[0] )    
    ds=ds_flat


x=ds.epwp_max.values
cumu,xs=auti.cumul_big_to_small( x, plot_it=True )

fracs=[0.995,0.90,0.50,0.25,0.125,0.0625]
thresh_fracs=[]
thresholds=[]
N_events=[]
Total_events=len(xs)
print( f"There are a total of {Total_events} events in {lon_range}X{lat_range}, exclude orography={exclude_orography} " )
for b in fracs:
    xoo=np.argmin( np.abs(cumu-b) )
    print(f" fraction={100*b:5.2f}% of total epwp is in events with epwp > {xs[xoo]:.5f}. Carried by N={len(xs[xoo:]):6d} events or {100*len(xs[xoo:])/len(xs):5.2f}%  ")
    thresh_fracs.append(xs[xoo])
    thresholds.append( np.array([ xs[xoo],1.e6] ) )
    N_events.append( len(xs[xoo:]) )
print(f" Threshholds {thresholds}")

In [ ]:
El=[]

print( f"This run use dycore={A.dycore}")

if A.dycore == 'MPAS':
    ## for MPAS 3km
    big_window=[3,5,5]
    lil_window=[3,2,2]
    
elif A.dycore == 'SE':
    # for ne240
    big_window=[2,5,5]
    lil_window=[2,2,2]


ithr=0
for thresh in thresholds:
    second_thresh=None #second_thresholds[ithr]
    ds =euti.make_ds(fld=A.rho_epwp[:,:,:,:], lon=lon, lat=lat, zlev=zlev, time=time, 
                     thresh=thresh,second_thresh=second_thresh,zlev_event=zlev_event, 
                     lat_range=lat_range, lon_range=lon_range)
    

    # get shape of varaiables
    nt,nz,ny,nx = np.shape( A.u )
    
    htopo_t = np.tile(A.htopo[None, :, :], ( nt, 1, 1))
    htopo_4D_x , time4D,lat4D,lon4D  = auti.cube4D_ds( event_ds=ds, aa=htopo_t , lon=lon, lat=lat, window=[0,5,5] , TZHkey='tyx', lat_range=[-999,999], lon_range=[-999,999] )
    
    htopo_MMM = auti.collapseSpace( htopo_4D_x , TZHkey='etyx')
    htopo_super_max = htopo_MMM[2].max( axis=1 )
    
    
    ########################################
    # Exclude events with topography nearby
    ########################################
    if (exclude_orography==True):
        flat=np.where(htopo_super_max<1.e-12 ) #0.0001)
        flat[0].shape
        ds_flat=ds.isel( index=flat[0] )    
        ds=ds_flat
    
    E_ = {'ds':ds }
    E_['threshold']=thresh
    if second_thresh is not None:
        E_["second_threshold"] = second_thresh        
    E_['exclude_orography'] = exclude_orography
    E_['lat_range']=lat_range
    E_['lon_range']=lon_range
    E_['N_events'] = ds.sizes['index']
    E_['Frac_of_total_epwp'] = fracs[ithr]

    window=lil_window # [3,2,2]
    #window=[6,2,2] # MPAS results are 3-hourly ...
    if ds.sizes['index'] < 75_000:
        window = big_window #[3,5,5] #[6,5,5]
        print( f"Big window ")
    #window=[6,5,5] # MPAS results are 3-hourly ...
    
    
    
    precl_4D , time4D,lat4D,lon4D  = auti.cube4D_ds( event_ds=ds, aa=A.precl , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    htopo_4D , time4D,lat4D,lon4D  = auti.cube4D_ds( event_ds=ds, aa=htopo_t , lon=lon, lat=lat, window=window , TZHkey='tyx', lat_range=lat_range, lon_range=lon_range )
    zeta_4D, time4D,lat4D,lon4D = auti.cube4D_ds( event_ds=ds, aa=A.zeta , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    tilt_4D, time4D,lat4D,lon4D = auti.cube4D_ds( event_ds=ds, aa=A.tilt , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    fgf_4D, time4D,lat4D,lon4D  = auti.cube4D_ds( event_ds=ds, aa=A.fgf , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    epwp_4D, time4D,lat4D,lon4D = auti.cube4D_ds( event_ds=ds, aa=A.rho_epwp , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    upwp_4D, time4D,lat4D,lon4D = auti.cube4D_ds( event_ds=ds, aa=A.rho_upwp , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    vpwp_4D, time4D,lat4D,lon4D = auti.cube4D_ds( event_ds=ds, aa=A.rho_vpwp , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    u_4D, time4D,lat4D,lon4D    = auti.cube4D_ds( event_ds=ds, aa=A.u , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    v_4D, time4D,lat4D,lon4D    = auti.cube4D_ds( event_ds=ds, aa=A.v , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    th_4D, time4D,lat4D,lon4D   = auti.cube4D_ds( event_ds=ds, aa=A.th , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )

    E_['time4D'], E_['lat4D'], E_['lon4D'] = time4D,lat4D,lon4D
    E_['u_4D'], E_['v_4D'], E_['htopo_4D'] = u_4D,v_4D,htopo_4D
    E_['zeta_4D'], E_['tilt_4D'] , E_['fgf_4D']  = zeta_4D,tilt_4D,fgf_4D
    E_['upwp_4D'], E_['vpwp_4D'] , E_['epwp_4D']  = upwp_4D,vpwp_4D,epwp_4D
    E_['precl_4D'], E_['th_4D'] = precl_4D,th_4D

    E = AttrDict( E_ )
    El.append(E)
    print( f" fininshed threshold = {thresh}. N events: = {ds.sizes['index']} " )
    ithr=ithr+1


In [ ]:
zetalv=1.5e-5*np.linspace(-6,6,num=13)
ulv=np.linspace(-60,60,num=27)
thlv=np.concatenate( (270.+np.arange(11)*10 , 380.+ np.arange(11)*20) )   #np.linspace(270,600,num=27)
mflv=[0.001,0.002,.005, .01, .02]
Epls=El #[El[0], El[1], El[2], El[3] ] #, Eco_0 ]
print(thlv)
nxplo,nyplo=len( Epls ),1
fig,axs=plt.subplots( nyplo,nxplo , figsize=(nxplo*7+1,nyplo*8) )
axs=axs.flatten()
p=0

for Epl in Epls:
    nv,nt_v,nz_v,ny_v,nx_v = np.shape( Epl.zeta_4D )
    delta_time=3

    ax=axs[p]
    Epl_vv=euti.avg_over_v(Epl)
    zeta_poo=Epl_vv.zeta_4D.mean(axis=3)
    u_poo=Epl_vv.u_4D.mean(axis=3)
    th_poo=Epl_vv.th_4D.mean(axis=3)
    epwp_poo=Epl_vv.epwp_4D.mean(axis=3)
    colo = ax.contourf( np.arange(ny_v), zlev, zeta_poo[nt_v-1,:,:], cmap='bwr' , levels=zetalv)
    lin1 = ax.contour( np.arange(ny_v), zlev, u_poo[nt_v-1,:,:] , levels=ulv)
    ax.clabel(lin1, inline=True, fontsize=8, fmt='%1.0f')
    lin2 = ax.contour( np.arange(ny_v), zlev, th_poo[nt_v-1,:,:] , levels=thlv, colors='red')
    ax.clabel(lin2, inline=True, fontsize=8, fmt='%1.0f')
    lin3 = ax.contour( np.arange(ny_v), zlev, epwp_poo[nt_v-1,:,:] , levels=mflv, colors='black')
    ax.clabel(lin3, inline=True, fontsize=8, fmt='%1.3f')
    ax.set_ylim(0,20_000)
    ax.set_title( f"N={Epl.N_events} of {Total_events} in {Epl.lon_range}X{Epl.lat_range}, \n exclude orography={Epl.exclude_orography}, \n epwp_thresh={Epl.threshold[0]:.4g} to {Epl.threshold[1]:.2g} \n fraction of total epwp {100*Epl.Frac_of_total_epwp:.2f}% " )
    p=p+1

cax = fig.add_axes([0.15, 0.02, 0.70, 0.03])
cbar = fig.colorbar(colo, cax=cax, orientation='horizontal')
cbar.set_label(f"vorticity  s{r'$^{-1}$' }")
"""
cax = fig.add_axes([0.15, -0.1, 0.70, 0.03])
cbar = fig.colorbar(linc, cax=cax, orientation='horizontal')
cbar.set_label('Some quantity')
"""

In [ ]:
zetalv=0.75e-7*np.linspace(0,6,num=31)
zetalv=1.00e-7*np.linspace(0,6,num=31)
ulv=np.linspace(-60,60,num=27)
thlv=np.concatenate( (270.+np.arange(11)*10 , 380.+ np.arange(11)*20) )   #np.linspace(270,600,num=27)
Epls=El  #[El[0], El[1], El[2], El[3] ] #, Eco_0 ]
print(thlv)
nxplo,nyplo=len( Epls ),1
fig,axs=plt.subplots( nyplo,nxplo , figsize=(nxplo*7+1,nyplo*8) )
axs=axs.flatten()
p=0

for Epl in Epls:
    nv,nt_v,nz_v,ny_v,nx_v = np.shape( Epl.zeta_4D )
    delta_time=3

    ax=axs[p]
    Epl_vv=euti.avg_over_v(Epl)
    tilt_poo=Epl_vv.tilt_4D.mean(axis=3)
    u_poo=Epl_vv.u_4D.mean(axis=3)
    th_poo=Epl_vv.th_4D.mean(axis=3)
    colo = ax.contourf( np.arange(ny_v), zlev, tilt_poo[nt_v-1,:,:], cmap='inferno' , levels=zetalv)
    lin1 = ax.contour( np.arange(ny_v), zlev, u_poo[nt_v-1,:,:] , levels=ulv)
    ax.clabel(lin1, inline=True, fontsize=8, fmt='%1.0f')
    lin2 = ax.contour( np.arange(ny_v), zlev, th_poo[nt_v-1,:,:] , levels=thlv, colors='red')
    ax.clabel(lin2, inline=True, fontsize=8, fmt='%1.0f')
    ax.set_ylim(0,20_000)
    ax.set_title( f"N={Epl.N_events} of {Total_events} in {Epl.lon_range}X{Epl.lat_range}, \n exclude orography={Epl.exclude_orography}, \n epwp_thresh={Epl.threshold[0]:.4g} to {Epl.threshold[1]:.2g} \n fraction of total epwp {100*Epl.Frac_of_total_epwp:.2f}% " )
    p=p+1

cax = fig.add_axes([0.15, 0.02, 0.70, 0.03])
cbar = fig.colorbar(colo, cax=cax, orientation='horizontal')
cbar.set_label(f"Tilting  s{r'$^{-2}$' }")
"""
cax = fig.add_axes([0.15, -0.1, 0.70, 0.03])
cbar = fig.colorbar(linc, cax=cax, orientation='horizontal')
cbar.set_label('Some quantity')
"""

In [ ]:
zetalv=1.5e-15*np.linspace(-6,6,num=31)
ulv=np.linspace(-60,60,num=27)
thlv=np.concatenate( (270.+np.arange(11)*10 , 380.+ np.arange(11)*20) )   #np.linspace(270,600,num=27)
Epls=El #, Eco_0 ]
print(thlv)
nxplo,nyplo=len( Epls ),1
fig,axs=plt.subplots( nyplo,nxplo , figsize=(nxplo*7+1,nyplo*8) )
axs=axs.flatten()
p=0

for Epl in Epls:
    nv,nt_v,nz_v,ny_v,nx_v = np.shape( Epl.zeta_4D )
    delta_time=3

    ax=axs[p]
    Epl_vv=euti.avg_over_v(Epl)
    fgf_poo=Epl_vv.fgf_4D.mean(axis=3)
    u_poo=Epl_vv.u_4D.mean(axis=3)
    th_poo=Epl_vv.th_4D.mean(axis=3)
    colo = ax.contourf( np.arange(ny_v), zlev, fgf_poo[nt_v-1,:,:], cmap='bwr' , levels=zetalv)
    lin1 = ax.contour( np.arange(ny_v), zlev, u_poo[nt_v-1,:,:] , levels=ulv)
    ax.clabel(lin1, inline=True, fontsize=8, fmt='%1.0f')
    lin2 = ax.contour( np.arange(ny_v), zlev, th_poo[nt_v-1,:,:] , levels=thlv, colors='red')
    ax.clabel(lin2, inline=True, fontsize=8, fmt='%1.0f')
    ax.set_ylim(0,20_000)
    ax.set_title( f"N={Epl.N_events} of {Total_events} in {Epl.lon_range}X{Epl.lat_range}, \n exclude orography={Epl.exclude_orography}, \n epwp_thresh={Epl.threshold[0]:.4g} to {Epl.threshold[1]:.2g} \n fraction of total epwp {100*Epl.Frac_of_total_epwp:.2f}% " )
    p=p+1

cax = fig.add_axes([0.15, 0.02, 0.70, 0.03])
cbar = fig.colorbar(colo, cax=cax, orientation='horizontal')
cbar.set_label(f"Frontogenesis  s{r'$^{-2}$' }")
"""
cax = fig.add_axes([0.15, -0.1, 0.70, 0.03])
cbar = fig.colorbar(linc, cax=cax, orientation='horizontal')
cbar.set_label('Some quantity')
"""

In [ ]:
print('poopypants')

In [ ]:

ds=El[2].ds
print(int(ds.itime.values.max()) + 1)

ds_drop=ds.drop_vars( ['lat','lon','zlev'] )
ds

In [ ]:
event_list=euti.ds_to_event_list( ds_drop )

In [ ]:
# event_list[t] must contain dicts with 't' key added
# (as we discussed earlier when adding timestep to each event)
importlib.reload( euti )

tracks = euti.track_events(
    event_list    = event_list,
    lat           = lat,
    lon           = lon,
    dt_hours      = 3.0,
    max_speed_kmh = 180.0,    # ~Southern Ocean cyclone speed
    min_lifetime  = 1,       # require at least 2 timesteps = 6 hours
)

_ = euti.plot_track_statistics_2(tracks, dt_hours=3.0)

In [ ]:
80_000./3600.

In [ ]:
A.rho_epwp.shape

In [ ]:
lfes=tracks[0]['lifetime']

tracks[0].keys()

In [ ]:
lifetimes  = np.array([tr['lifetime']   for tr in tracks])
speeds     = np.array([tr['mean_speed'] for tr in tracks])
maxlife    = np.max(lifetimes)+1
ntracks    = len(lifetimes)


shape = (ntracks, maxlife)

ixs   = np.full(shape, -1, dtype=int)
iys   = np.full(shape, -1, dtype=int)
times = np.full(shape, -1, dtype=int)

itrk=0
for tr in tracks:
    length=len(tr['times'])
    times[itrk,0:length] = tr['times']
    itrk=itrk+1

#vel_y_all  = np.concatenate([tr['vel_y_kmh'][1:] for tr in tracks])
#vel_x_all  = np.concatenate([tr['vel_x_kmh'][1:] for tr in tracks])
#ixs        = np.concatenate([tr['ix'] for tr in tracks])
#iys        = np.concatenate([tr['iy'] for tr in tracks])
#times      = np.concatenate([tr['times'] for tr in tracks])


In [ ]:
print( times[400,:] )

In [ ]:
plt.plot(lifetimes,'o')
plt.xlim(790,800)

In [ ]:
ixs.shape

In [ ]:
importlib.reload(euti)
trx=euti.trackarrays( tracks, lon=A.lon,lat=A.lat )

In [ ]:
itx=791
plt.scatter( trx.lons[itx,:]  , trx.lats[itx,:] )

print( trx.times[itx,:] )

In [ ]:
z0=np.argmin( np.abs( zlev-0.))
z0p5=np.argmin( np.abs( zlev-500))
z1=np.argmin( np.abs( zlev-1000.))
z3=np.argmin( np.abs( zlev-3000.))
z5=np.argmin( np.abs( zlev-5000.))
z6=np.argmin( np.abs( zlev-6000.))
z7=np.argmin( np.abs( zlev-7000.))
z10=np.argmin( np.abs( zlev-10000.))
z11=np.argmin( np.abs( zlev-11000.))
z12=np.argmin( np.abs( zlev-12000.))
z15=np.argmin( np.abs( zlev-15000.))

tim=144
oo=np.where( trx.times == tim )


A.lat.shape
plt.contourf(lon,lat, np.log(A.epwp[tim,z15,:,:]+1e-6)  )
plt.contour(lon,lat, A.htopo ,levels=[1.e-12,0.1,1,10,100,1000])

for n in np.arange(len(oo[0]) ):
    i,j=oo[0][n] , oo[1][n]
    plt.scatter( trx.lons[i,j]  , trx.lats[i,j] , color='white')

#itx=791
#plt.scatter( trx.lons[itx,:]  , trx.lats[itx,:] , color='white')

In [ ]:
print(oo[0][1])

In [ ]:
for n in np.arange(len(oo[0])):
    print(n)
    i,j=oo[0][n] , oo[1][n]
    print(i,j)

In [ ]:
tr=tracks[402]
loop=np.zeros(10)
print(lon[ np.array(tr['ix'][0:1])]  )
loop[0:1] = lon[ np.array(tr['ix'][0:1])] 